## Cell 1 — Import library

In [1]:
import re
import pandas as pd
from pathlib import Path
import fitz  # PyMuPDF

## Cell 2 — Tentukan folder project dan folder label

In [2]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

LABEL_DIR = PROJECT_ROOT / "data" / "label"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Label dir:", LABEL_DIR)
print("Processed dir:", PROCESSED_DIR)

Project root: e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL
Label dir: e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\data\label
Processed dir: e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\data\processed


## Cell 3 — Cek file PDF nilai

In [3]:
pdf_files = sorted(LABEL_DIR.glob("*.pdf"))

print("Jumlah file PDF ditemukan:", len(pdf_files))

for file in pdf_files:
    print("-", file.name)

Jumlah file PDF ditemukan: 2
- IF-49-02_Kalkulus.pdf
- IF-49-03_Kalkulus.pdf


## Cell 4 — Fungsi membaca teks PDF

In [4]:
def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    text_pages = []

    for page_number, page in enumerate(doc, start=1):
        text = page.get_text()
        text_pages.append(text)

    doc.close()
    return "\n".join(text_pages)

## Cell 5 — Tes baca PDF pertama

In [5]:
sample_text = extract_text_from_pdf(pdf_files[0])

print(sample_text[:3000])

Telkom University
Jl.Telekomunikasi No.1, Terusan Buah Batu
Bandung 40257
Indonesia
No.
NIM
Nama
Kehadiran
SUB-CLO-
03-1-1-KO
(25%)
SUB-CLO-
03-1-2-KO
(25%)
SUB-CLO-
03-1-3-KO
(25%)
SUB-CLO-
03-1-4-KO
(25%)
Nilai
Total
Nilai
Indeks
1
103012400054
MUHAMMAD FAZLI AULIA
72.22%
9.86
19.83
0
14.62
11.08
E
2
103012400409
FACHRI RAZAQ KHALISH
50%
25.5
0
0
0
6.38
E
3
103012500035
RIZKY ALVINO PUTRA
100%
57.5
74.46
62.88
57
62.96
BC
4
103012500050
ALIF ZAIDAN ZIDNA FANN
100%
60.55
65.56
75.91
69
67.76
B
5
103012500054
RAFI AJRUR MUZAKKI
94.44%
60.23
64.03
58.26
67.86
62.6
BC
6
103012500068
RAIS GALANG SAPUTRA
100%
64.05
75
64
70.38
68.36
B
7
103012500076
MUHAMMAD GHIFRAN
FEBRIAN
100%
53.16
59.71
50.17
56.32
54.84
C
8
103012500079
MEHTAR NAFISA KRISNU
94.44%
65.14
79.73
64.89
67.16
69.23
B
9
103012500082
MAULANA ADE
FATHURROZAQI
100%
69.65
72.06
62.8
69.46
68.49
B
10
103012500108
ANDI ALRALY PASHA
RAHADHAN
94.44%
46.46
53
46.9
54.94
50.33
C
11
103012500122
RAYNISA ZAHRA
100%
76.21
80.4
73.72
70.

## Cell 6 — Tes baca dua PDF dan simpan teks sementara

In [6]:
pdf_texts = {}

for file in pdf_files:
    pdf_texts[file.name] = extract_text_from_pdf(file)
    print(file.name, "panjang teks:", len(pdf_texts[file.name]))

IF-49-02_Kalkulus.pdf panjang teks: 4615
IF-49-03_Kalkulus.pdf panjang teks: 4609


## Cell 7 — Lihat pola baris yang mengandung NIM

In [7]:
for file_name, text in pdf_texts.items():
    print("\n===", file_name, "===")
    
    lines = text.splitlines()
    nim_lines = [line for line in lines if re.search(r"103012\d+", line)]
    
    for line in nim_lines[:10]:
        print(line)


=== IF-49-02_Kalkulus.pdf ===
103012400054
103012400409
103012500035
103012500050
103012500054
103012500068
103012500076
103012500079
103012500082
103012500108

=== IF-49-03_Kalkulus.pdf ===
103012400053
103012400245
103012440008
103012500008
103012500020
103012500046
103012500063
103012500064
103012500075
103012500077


## Cell 8 — Cek apakah file label manual sudah ada

In [8]:
LABEL_MANUAL_PATH = LABEL_DIR / "label_performance_manual.csv"

print("Path file label manual:")
print(LABEL_MANUAL_PATH)

print("\nFile sudah ada:", LABEL_MANUAL_PATH.exists())

Path file label manual:
e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\data\label\label_performance_manual.csv

File sudah ada: True


## Cell 9 — Buat template CSV kosong

In [9]:
label_template = pd.DataFrame(columns=[
    "nim",
    "nama",
    "kelas",
    "nilai_total",
    "nilai_indeks",
    "label_performance"
])

label_template.to_csv(LABEL_MANUAL_PATH, index=False)

print("Template label berhasil dibuat di:")
print(LABEL_MANUAL_PATH)

Template label berhasil dibuat di:
e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\data\label\label_performance_manual.csv


## Cell 10 — Baca CSV label manual

In [10]:
label_df = pd.read_csv(LABEL_MANUAL_PATH)

print("Ukuran label_df:", label_df.shape)
display(label_df.head())

Ukuran label_df: (0, 6)


,nim,nama,kelas,nilai_total,nilai_indeks,label_performance


## Cell 8 — Fungsi parsing tabel nilai dari PDF

In [11]:
def infer_kelas_from_filename(file_name):
    if "49-02" in file_name:
        return "IF-49-02"
    elif "49-03" in file_name:
        return "IF-49-03"
    else:
        return "UNKNOWN"


def parse_grade_pdf_by_position(pdf_path):
    kelas = infer_kelas_from_filename(pdf_path.name)
    doc = fitz.open(pdf_path)
    records = []

    for page in doc:
        words = page.get_text("words")

        # Ubah tuple word menjadi dictionary agar lebih mudah dibaca
        word_items = []
        for word in words:
            x0, y0, x1, y1, text, block, line, word_no = word
            word_items.append({
                "x0": x0,
                "y0": y0,
                "x1": x1,
                "y1": y1,
                "text": text
            })

        # Cari semua kata yang bentuknya NIM
        nim_words = [
            w for w in word_items
            if re.fullmatch(r"103012\d+", w["text"])
        ]

        for nim_word in nim_words:
            y = nim_word["y0"]
            nim = nim_word["text"]

            # NIM pada tabel berada sekitar x=57.
            # Ini untuk menghindari NIM yang mungkin muncul di luar tabel.
            if not (50 <= nim_word["x0"] <= 100):
                continue

            # Ambil kata-kata yang berada pada baris yang sama dengan NIM
            same_row_words = [
                w for w in word_items
                if abs(w["y0"] - y) <= 1.5
            ]

            # Ambil nomor urut
            no_words = [
                w for w in same_row_words
                if w["x0"] < 55 and re.fullmatch(r"\d+", w["text"])
            ]
            no = int(no_words[0]["text"]) if no_words else None

            # Ambil nama.
            # Nama berada di area x sekitar 112 sampai 225.
            # Nama panjang kadang pecah ke baris atas/bawah, jadi toleransi y dibuat lebih lebar.
            name_words = [
                w for w in word_items
                if 112 <= w["x0"] <= 225 and abs(w["y0"] - y) <= 7.0
            ]

            nama = " ".join([
                w["text"]
                for w in sorted(name_words, key=lambda item: (item["y0"], item["x0"]))
            ])

            # Ambil nilai total.
            # Di tabel PDF, nilai total berada di area x sekitar 500.
            total_words = [
                w for w in same_row_words
                if 488 <= w["x0"] <= 525 and re.fullmatch(r"\d+(\.\d+)?", w["text"])
            ]
            nilai_total = float(total_words[0]["text"]) if total_words else None

            # Ambil indeks nilai.
            # Indeks berada di area kanan tabel.
            indeks_words = [
                w for w in same_row_words
                if 535 <= w["x0"] <= 555 and re.fullmatch(r"A|AB|B|BC|C|D|E|T", w["text"])
            ]
            nilai_indeks = indeks_words[0]["text"] if indeks_words else None

            records.append({
                "no": no,
                "nim": nim,
                "nama": nama,
                "kelas": kelas,
                "nilai_total": nilai_total,
                "nilai_indeks": nilai_indeks
            })

    doc.close()

    df = pd.DataFrame(records)
    df = df.sort_values(["kelas", "no"]).reset_index(drop=True)

    return df

## Cell 9 — Parse dua PDF nilai

In [12]:
label_dfs = []

for pdf_file in pdf_files:
    parsed_df = parse_grade_pdf_by_position(pdf_file)
    label_dfs.append(parsed_df)

    print(pdf_file.name)
    print("Jumlah baris terbaca:", len(parsed_df))
    display(parsed_df.head())
    print()

label_df = pd.concat(label_dfs, ignore_index=True)

print("Total label terbaca:", len(label_df))
display(label_df.head(10))
display(label_df.tail(10))

IF-49-02_Kalkulus.pdf
Jumlah baris terbaca: 45


,no,nim,nama,kelas,nilai_total,nilai_indeks
0,1,103012400054,MUHAMMAD FAZLI AULIA,IF-49-02,11.08,E
1,2,103012400409,FACHRI RAZAQ KHALISH,IF-49-02,6.38,E
2,3,103012500035,RIZKY ALVINO PUTRA,IF-49-02,62.96,BC
3,4,103012500050,ALIF ZAIDAN ZIDNA FANN,IF-49-02,67.76,B
4,5,103012500054,RAFI AJRUR MUZAKKI,IF-49-02,62.60,BC



IF-49-03_Kalkulus.pdf
Jumlah baris terbaca: 44


,no,nim,nama,kelas,nilai_total,nilai_indeks
0,1,103012400053,MUHAMMAD AKRAM,IF-49-03,2.83,E
1,2,103012400245,TOM HADIWIJOYO,IF-49-03,18.90,E
2,3,103012440008,MUHAMMAD HADYAN GHOSSAN SUSILO,IF-49-03,65.93,B
3,4,103012500008,NABIL COZADI ALFARIZI,IF-49-03,62.27,BC
4,5,103012500020,MUHAMMAD FADHIL,IF-49-03,75.05,AB



Total label terbaca: 89


,no,nim,nama,kelas,nilai_total,nilai_indeks
0,1,103012400054,MUHAMMAD FAZLI AULIA,IF-49-02,11.08,E
1,2,103012400409,FACHRI RAZAQ KHALISH,IF-49-02,6.38,E
2,3,103012500035,RIZKY ALVINO PUTRA,IF-49-02,62.96,BC
3,4,103012500050,ALIF ZAIDAN ZIDNA FANN,IF-49-02,67.76,B
4,5,103012500054,RAFI AJRUR MUZAKKI,IF-49-02,62.60,BC
5,6,103012500068,RAIS GALANG SAPUTRA,IF-49-02,68.36,B
6,7,103012500076,MUHAMMAD GHIFRAN FEBRIAN,IF-49-02,54.84,C
7,8,103012500079,MEHTAR NAFISA KRISNU,IF-49-02,69.23,B
8,9,103012500082,MAULANA ADE FATHURROZAQI,IF-49-02,68.49,B
9,10,103012500108,ANDI ALRALY PASHA RAHADHAN,IF-49-02,50.33,C


,no,nim,nama,kelas,nilai_total,nilai_indeks
79,35,103012500403,ARZELIN SUDRAJAT,IF-49-03,71.38,B
80,36,103012530001,REYHAN FADHILAH AHMADHUSEIN,IF-49-03,47.64,D
81,37,103012530011,NATHANIEL DARIUS AHMAD RAJA GUKGUK,IF-49-03,69.70,B
82,38,103012530024,FRANSISKUS BRAMA WIJAYA,IF-49-03,62.61,BC
83,39,103012530032,ALFITO MAULANA,IF-49-03,67.24,B
84,40,103012530042,ANDREANUS SONDIKA BUTAR BUTAR,IF-49-03,79.89,AB
85,41,103012530044,ASHIFA SUKMANING CAHYA,IF-49-03,60.10,BC
86,42,103012530064,MOCHAMAD FARREL FABIANSYAH,IF-49-03,75.13,AB
87,43,103012530069,ANANDA PUTRA PAMUNGKAS,IF-49-03,76.17,AB
88,44,103012530076,TEUKU FAWWAZ ALWANDA,IF-49-03,56.42,C


## Cell 10 — Validasi hasil parsing

In [13]:
print("Ukuran label_df:", label_df.shape)

print("\nJumlah missing value:")
display(label_df.isna().sum())

print("\nJumlah NIM duplikat:")
print(label_df["nim"].duplicated().sum())

print("\nDistribusi kelas:")
display(label_df["kelas"].value_counts())

print("\nDistribusi nilai indeks:")
display(label_df["nilai_indeks"].value_counts().sort_index())

print("\nDistribusi nilai indeks per kelas:")
display(pd.crosstab(label_df["kelas"], label_df["nilai_indeks"]))

Ukuran label_df: (89, 6)

Jumlah missing value:


no              0
nim             0
nama            0
kelas           0
nilai_total     0
nilai_indeks    0
dtype: int64


Jumlah NIM duplikat:
0

Distribusi kelas:


kelas
IF-49-02    45
IF-49-03    44
Name: count, dtype: int64


Distribusi nilai indeks:


nilai_indeks
A      5
AB    15
B     43
BC    11
C      9
D      2
E      4
Name: count, dtype: int64


Distribusi nilai indeks per kelas:


nilai_indeks,A,AB,B,BC,C,D,E
kelas,,,,,,,
IF-49-02,1,6,23,5,7,1,2
IF-49-03,4,9,20,6,2,1,2


## Cell 11 — Buat label performa akademik

In [14]:
def map_performance_label(nilai_indeks):
    if nilai_indeks in ["A", "AB"]:
        return "Tinggi"
    elif nilai_indeks in ["B", "BC"]:
        return "Sedang"
    elif nilai_indeks in ["C", "D", "E"]:
        return "Rendah"
    else:
        return "Tidak_Diketahui"


label_df["label_performance"] = label_df["nilai_indeks"].apply(map_performance_label)

display(label_df.head(10))

print("Distribusi label performance:")
display(label_df["label_performance"].value_counts())

,no,nim,nama,kelas,nilai_total,nilai_indeks,label_performance
0,1,103012400054,MUHAMMAD FAZLI AULIA,IF-49-02,11.08,E,Rendah
1,2,103012400409,FACHRI RAZAQ KHALISH,IF-49-02,6.38,E,Rendah
2,3,103012500035,RIZKY ALVINO PUTRA,IF-49-02,62.96,BC,Sedang
3,4,103012500050,ALIF ZAIDAN ZIDNA FANN,IF-49-02,67.76,B,Sedang
4,5,103012500054,RAFI AJRUR MUZAKKI,IF-49-02,62.60,BC,Sedang
5,6,103012500068,RAIS GALANG SAPUTRA,IF-49-02,68.36,B,Sedang
6,7,103012500076,MUHAMMAD GHIFRAN FEBRIAN,IF-49-02,54.84,C,Rendah
7,8,103012500079,MEHTAR NAFISA KRISNU,IF-49-02,69.23,B,Sedang
8,9,103012500082,MAULANA ADE FATHURROZAQI,IF-49-02,68.49,B,Sedang
9,10,103012500108,ANDI ALRALY PASHA RAHADHAN,IF-49-02,50.33,C,Rendah


Distribusi label performance:


label_performance
Sedang    54
Tinggi    20
Rendah    15
Name: count, dtype: int64

## Cell 12 — Simpan label ke data processed

In [15]:
LABEL_OUTPUT_PATH = PROCESSED_DIR / "02_label_performance.csv"

label_df.to_csv(LABEL_OUTPUT_PATH, index=False)

print("Label performance berhasil disimpan ke:")
print(LABEL_OUTPUT_PATH)

Label performance berhasil disimpan ke:
e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\data\processed\02_label_performance.csv


## Cell 13 — Baca ulang file hasil simpan

In [16]:
label_check = pd.read_csv(LABEL_OUTPUT_PATH)

print("Ukuran file label yang dibaca ulang:", label_check.shape)
display(label_check.head())
display(label_check["label_performance"].value_counts())

Ukuran file label yang dibaca ulang: (89, 7)


,no,nim,nama,kelas,nilai_total,nilai_indeks,label_performance
0,1,103012400054,MUHAMMAD FAZLI AULIA,IF-49-02,11.08,E,Rendah
1,2,103012400409,FACHRI RAZAQ KHALISH,IF-49-02,6.38,E,Rendah
2,3,103012500035,RIZKY ALVINO PUTRA,IF-49-02,62.96,BC,Sedang
3,4,103012500050,ALIF ZAIDAN ZIDNA FANN,IF-49-02,67.76,B,Sedang
4,5,103012500054,RAFI AJRUR MUZAKKI,IF-49-02,62.60,BC,Sedang


label_performance
Sedang    54
Tinggi    20
Rendah    15
Name: count, dtype: int64

## Cell 12 — Rapikan tipe data sebelum disimpan

In [17]:
label_df["nim"] = label_df["nim"].astype(str)
label_df["nama"] = label_df["nama"].str.strip()
label_df["kelas"] = label_df["kelas"].str.strip()
label_df["nilai_indeks"] = label_df["nilai_indeks"].str.strip()
label_df["label_performance"] = label_df["label_performance"].str.strip()

display(label_df.head())
print(label_df.dtypes)

,no,nim,nama,kelas,nilai_total,nilai_indeks,label_performance
0,1,103012400054,MUHAMMAD FAZLI AULIA,IF-49-02,11.08,E,Rendah
1,2,103012400409,FACHRI RAZAQ KHALISH,IF-49-02,6.38,E,Rendah
2,3,103012500035,RIZKY ALVINO PUTRA,IF-49-02,62.96,BC,Sedang
3,4,103012500050,ALIF ZAIDAN ZIDNA FANN,IF-49-02,67.76,B,Sedang
4,5,103012500054,RAFI AJRUR MUZAKKI,IF-49-02,62.60,BC,Sedang


no                     int64
nim                      str
nama                     str
kelas                    str
nilai_total          float64
nilai_indeks             str
label_performance        str
dtype: object


## Cell 13 — Simpan label ke data processed

In [18]:
LABEL_OUTPUT_PATH = PROCESSED_DIR / "02_label_performance.csv"

label_df.to_csv(LABEL_OUTPUT_PATH, index=False)

print("Label performance berhasil disimpan ke:")
print(LABEL_OUTPUT_PATH)

Label performance berhasil disimpan ke:
e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\data\processed\02_label_performance.csv


## Cell 14 — Baca ulang file label

In [19]:
label_check = pd.read_csv(LABEL_OUTPUT_PATH, dtype={"nim": str})

print("Ukuran file label yang dibaca ulang:", label_check.shape)

display(label_check.head())
display(label_check.tail())

print("\nDistribusi label performance:")
display(label_check["label_performance"].value_counts())

Ukuran file label yang dibaca ulang: (89, 7)


,no,nim,nama,kelas,nilai_total,nilai_indeks,label_performance
0,1,103012400054,MUHAMMAD FAZLI AULIA,IF-49-02,11.08,E,Rendah
1,2,103012400409,FACHRI RAZAQ KHALISH,IF-49-02,6.38,E,Rendah
2,3,103012500035,RIZKY ALVINO PUTRA,IF-49-02,62.96,BC,Sedang
3,4,103012500050,ALIF ZAIDAN ZIDNA FANN,IF-49-02,67.76,B,Sedang
4,5,103012500054,RAFI AJRUR MUZAKKI,IF-49-02,62.60,BC,Sedang


,no,nim,nama,kelas,nilai_total,nilai_indeks,label_performance
84,40,103012530042,ANDREANUS SONDIKA BUTAR BUTAR,IF-49-03,79.89,AB,Tinggi
85,41,103012530044,ASHIFA SUKMANING CAHYA,IF-49-03,60.10,BC,Sedang
86,42,103012530064,MOCHAMAD FARREL FABIANSYAH,IF-49-03,75.13,AB,Tinggi
87,43,103012530069,ANANDA PUTRA PAMUNGKAS,IF-49-03,76.17,AB,Tinggi
88,44,103012530076,TEUKU FAWWAZ ALWANDA,IF-49-03,56.42,C,Rendah



Distribusi label performance:


label_performance
Sedang    54
Tinggi    20
Rendah    15
Name: count, dtype: int64

## Cell 15 — Simpan ringkasan label ke outputs/tables

In [20]:
TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

label_distribution = (
    label_df["label_performance"]
    .value_counts()
    .reset_index()
)

label_distribution.columns = ["label_performance", "jumlah_mahasiswa"]
label_distribution["persentase"] = (
    label_distribution["jumlah_mahasiswa"] / len(label_df) * 100
).round(2)

index_distribution = (
    label_df["nilai_indeks"]
    .value_counts()
    .sort_index()
    .reset_index()
)

index_distribution.columns = ["nilai_indeks", "jumlah_mahasiswa"]
index_distribution["persentase"] = (
    index_distribution["jumlah_mahasiswa"] / len(label_df) * 100
).round(2)

label_distribution.to_csv(TABLES_DIR / "02_distribution_label_performance.csv", index=False)
index_distribution.to_csv(TABLES_DIR / "02_distribution_nilai_indeks.csv", index=False)

display(label_distribution)
display(index_distribution)

print("Ringkasan label berhasil disimpan ke:")
print(TABLES_DIR)

,label_performance,jumlah_mahasiswa,persentase
0,Sedang,54,60.67
1,Tinggi,20,22.47
2,Rendah,15,16.85


,nilai_indeks,jumlah_mahasiswa,persentase
0,A,5,5.62
1,AB,15,16.85
2,B,43,48.31
3,BC,11,12.36
4,C,9,10.11
5,D,2,2.25
6,E,4,4.49


Ringkasan label berhasil disimpan ke:
e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\outputs\tables


## Kesimpulan Prepare Label Performance

Pada tahap ini, data nilai akademik mahasiswa dari kelas IF-49-02 dan IF-49-03 berhasil diekstraksi dari file PDF resmi. Hasil ekstraksi menghasilkan 89 data mahasiswa, terdiri dari 45 mahasiswa IF-49-02 dan 44 mahasiswa IF-49-03. Pemeriksaan kualitas data menunjukkan bahwa tidak terdapat missing value dan tidak terdapat NIM duplikat.

Nilai indeks akademik kemudian dipetakan menjadi tiga kategori performa, yaitu Rendah, Sedang, dan Tinggi. Indeks A dan AB dipetakan sebagai Tinggi, indeks B dan BC dipetakan sebagai Sedang, sedangkan indeks C, D, dan E dipetakan sebagai Rendah. Hasil akhir menunjukkan distribusi label performa sebanyak 54 mahasiswa pada kategori Sedang, 20 mahasiswa pada kategori Tinggi, dan 15 mahasiswa pada kategori Rendah.

File label performa akademik disimpan sebagai `data/processed/02_label_performance.csv` dan akan digunakan sebagai target pada tahap pemodelan machine learning.